# Soft Margins & Tuning C

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/svm/03-soft-margins

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Intuition — allow mistakes to gain robustness

Real data overlaps, so demanding a *perfect* separation (hard margin) is brittle or impossible. The
**soft-margin** SVM allows some points to violate the margin, paying a **hinge-loss** penalty for
each. The regularization parameter **C** sets the price of violations: **small C** buys a **wide
margin** by tolerating many violators (more regularized, higher bias); **large C** insists on
classifying everything, yielding a **narrow margin** that contorts around stragglers (lower bias,
higher variance, outlier-sensitive). Minimizing `½‖w‖² + C·Σ hinge` is convex, and we optimize it by
(sub)gradient descent, then validate against `sklearn`.

## Hinge-loss SVM by gradient descent

The soft-margin SVM is just hinge loss + L2, so we can train it with plain subgradient descent and watch C work.

In [ ]:
def make_data(n=80, overlap=1.2):
    X0 = np.random.randn(n // 2, 2) * overlap + [-1.5, -1]
    X1 = np.random.randn(n // 2, 2) * overlap + [1.5, 1]
    X = np.vstack([X0, X1]); y = np.array([-1] * (n // 2) + [1] * (n // 2))
    return X, y

X, y = make_data()

def train_svm(X, y, C, epochs=400, lr=0.01):
    w = np.zeros(2); b = 0.0
    for _ in range(epochs):
        margins = y * (X @ w + b)
        viol = margins < 1                     # only violators have gradient
        w -= lr * (w - C * (y[viol, None] * X[viol]).sum(axis=0))
        b -= lr * (-C * y[viol].sum())
    return w, b

## The same data at three prices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
for ax, C in zip(axes, [0.01, 1.0, 100.0]):
    w, b = train_svm(X, y, C)
    Z = (xx * w[0] + yy * w[1] + b)
    ax.contourf(xx, yy, np.sign(Z), levels=1, colors=['#6366f133', '#14b8a633'])
    for lv, ls in [(-1, ':'), (0, '-'), (1, ':')]:
        ax.contour(xx, yy, Z, levels=[lv], colors='white', linestyles=ls, linewidths=1.2)
    ax.scatter(*X[y == -1].T, c='#6366f1', s=18); ax.scatter(*X[y == 1].T, c='#14b8a6', s=18)
    margin = 2 / np.linalg.norm(w)
    ax.set_title(f'C = {C}   margin = {margin:.2f}')
plt.tight_layout(); plt.show()
# small C: wide margin, many violations tolerated
# large C: narrow margin, boundary contorts to satisfy stragglers

**What to notice:** the *same* data at three prices of C. **Small C (0.01)** gives the **widest**
margin and shrugs off the overlapping points; **large C (100)** narrows the margin and bends the
boundary to satisfy nearly every point. The margin width printed in each title shrinks as C grows —
C directly trades margin width against training errors.

## The library way — validate against `sklearn`

Our (sub)gradient-descent soft-margin SVM should reach essentially the same boundary as
`sklearn.svm.SVC(kernel='linear')` at the same C. The cell checks the accuracy matches and the weight
vectors point the same way.

In [ ]:
from sklearn.svm import SVC

w, b = train_svm(X, y, C=1.0)
sk = SVC(kernel='linear', C=1.0).fit(X, y)

our_acc = np.mean(np.sign(X @ w + b) == y)
sk_acc  = sk.score(X, y)
cos = (w @ sk.coef_[0]) / (np.linalg.norm(w) * np.linalg.norm(sk.coef_[0]))
print(f'our GD SVM accuracy: {our_acc:.3f} | sklearn: {sk_acc:.3f} | weight-direction cosine: {cos:.3f}')
assert abs(our_acc - sk_acc) < 0.05 and cos > 0.97, "our soft-margin SVM must match sklearn"
print('our (sub)gradient-descent SVM == sklearn SVC ✓')

**What to notice:** matching accuracy and a weight-direction cosine near 1 — our hand-rolled
hinge-loss gradient descent finds the same soft-margin hyperplane `sklearn`'s optimized solver does.
Different algorithm, same convex optimum.

## Count the violators

In [ ]:
for C in [0.01, 1.0, 100.0]:
    w, b = train_svm(X, y, C)
    xi = np.maximum(0, 1 - y * (X @ w + b))
    print(f'C={C:>6}:  margin={2/np.linalg.norm(w):5.2f}   '
          f'violators(ξ>0)={int((xi > 1e-9).sum()):2d}   misclassified(ξ>1)={int((xi > 1).sum()):2d}')

**What to notice:** counting the margin **violators** confirms the tradeoff quantitatively — small
C tolerates *many* points inside the margin (wide, forgiving), large C drives the count down (narrow,
strict). The violators are exactly the points with nonzero hinge loss, i.e. the ones the boundary
actually "feels."

## Gotchas & tradeoffs

- **C is the key hyperparameter** — tune it by cross-validation. Too large overfits (chases every
  point, including noise); too small underfits.
- **Large C is outlier-sensitive.** A single mislabeled/outlier point can drag a high-C boundary a
  long way, because every violation is expensive.
- **Scale features first.** Like all SVMs, the margin depends on distances — standardize.
- **Hinge loss has a kink.** It's non-differentiable at the margin, so training uses **subgradients**
  (as in the exercise) — the gradient is 0 for safely-classified points.

In [ ]:
# Large C reacts strongly to a single outlier; small C shrugs it off
np.random.seed(0)
X_out = np.vstack([X, [[7.0, -7.0]]])          # one outlier deep in the wrong class's territory
y_out = np.append(y, 1)
for C in [0.05, 100.0]:
    w0, _ = train_svm(X, y, C)
    w1, _ = train_svm(X_out, y_out, C)
    shift = np.linalg.norm(w1 - w0)
    print(f'C={C:>6}: boundary shift from adding one outlier = {shift:.3f}')
print('\n-> large C lets a single outlier move the boundary much more (less robust)')

**What to notice:** adding one outlier moves the **large-C** boundary far more than the small-C one —
because at high C every violation is costly, so the model distorts itself to accommodate even a single
bad point. Small C's wider margin absorbs the outlier gracefully. This is the robustness half of the
bias-variance tradeoff C controls.

## Key takeaways

- The **soft margin** allows margin violations (hinge loss), making the SVM work on overlapping,
  noisy data.
- **C** trades margin width against violations: small C → wide margin / more bias / robust; large C →
  narrow margin / less bias / outlier-sensitive.
- The objective `½‖w‖² + C·Σ hinge` is **convex**; (sub)gradient descent matches `sklearn`'s solver.
- Tune **C by cross-validation**, scale features, and remember only the **violators + support
  vectors** shape the boundary.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/svm/04-quiz).

**Try it:** add one extreme outlier with `X = np.vstack([X, [[-4, 3]]]); y = np.append(y, 1)` and retrain at each C. Watch large C reshape the whole boundary for one point while small C shrugs.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Count the violators

A point with margin $y \cdot s < 1$ is a **violator**; its slack $\xi = \max(0, 1 - y s)$ measures by how much. Implement both summaries — they're the two numbers the soft-margin objective trades off against margin width.

In [ ]:
def count_violations(y, scores):
    """How many points have margin y * s < 1."""
    y = np.asarray(y, dtype=float)
    scores = np.asarray(scores, dtype=float)

    # TODO(you): count entries with y * scores < 1
    return ...


def total_slack(y, scores):
    """Sum of slacks: sum of max(0, 1 - y*s)."""
    y = np.asarray(y, dtype=float)
    scores = np.asarray(scores, dtype=float)

    # TODO(you): sum the hinge values
    return ...

In [ ]:
# Checks — run me
y = np.array([1, 1, -1, -1, 1])
s = np.array([2.0, 0.5, -3.0, 0.2, -0.5])

assert count_violations(y, s) == 3, "margins 2, .5, 3, -.2, -.5 -> three below 1"
assert abs(total_slack(y, s) - (0.5 + 1.2 + 1.5)) < 1e-12, "slacks 0 + .5 + 0 + 1.2 + 1.5"
assert count_violations([1], [1.0]) == 0 and total_slack([1], [1.0]) == 0.0, "exactly on the margin: no slack"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def count_violations(y, scores):
    y = np.asarray(y, dtype=float)
    scores = np.asarray(scores, dtype=float)
    return int(np.sum(y * scores < 1))


def total_slack(y, scores):
    y = np.asarray(y, dtype=float)
    scores = np.asarray(scores, dtype=float)
    return float(np.sum(np.maximum(0.0, 1.0 - y * scores)))
```

</details>

### Exercise 2 — One subgradient step

The update rule the gradient-descent trainer above uses, for a single point:

$$\mathbf{g} = \begin{cases} \lambda \mathbf{w} - y\,\mathbf{x} & \text{if } y \,(\mathbf{w} \cdot \mathbf{x}) < 1 \quad \text{(violator)} \\ \lambda \mathbf{w} & \text{otherwise} \end{cases}
\qquad \mathbf{w} \leftarrow \mathbf{w} - \eta \, \mathbf{g}$$

Violators pull $\mathbf{w}$ toward $y\,\mathbf{x}$; satisfied points only feel weight decay. The checks verify both branches and that repeated steps drive a point's margin to $\approx 1$.

In [ ]:
def svm_subgrad_step(w, x, y, lam=0.1, lr=0.5):
    """One subgradient step of the hinge + L2 objective on a single point."""
    w = np.asarray(w, dtype=float)
    x = np.asarray(x, dtype=float)

    # TODO(you): pick the subgradient by whether y * (w . x) < 1
    if ...:
        g = ...
    else:
        g = ...

    return w - lr * g

In [ ]:
# Checks — run me
w_new = svm_subgrad_step(np.zeros(2), [1.0, 2.0], 1, lam=0.1, lr=0.5)
assert np.allclose(w_new, [0.5, 1.0]), "violator: w moves toward y*x by lr"

w_new = svm_subgrad_step(np.array([2.0, 0.0]), [1.0, 0.0], 1, lam=0.1, lr=0.5)
assert np.allclose(w_new, [1.9, 0.0]), "margin met (score 2): only weight decay lam*w"

w = np.array([0.5, 0.5])
for _ in range(200):
    w = svm_subgrad_step(w, [1.0, 1.0], 1, lam=0.1, lr=0.1)
assert np.dot(w, [1.0, 1.0]) >= 1 - 0.05, "training on one point drives its margin to ~1"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def svm_subgrad_step(w, x, y, lam=0.1, lr=0.5):
    w = np.asarray(w, dtype=float)
    x = np.asarray(x, dtype=float)
    if y * np.dot(w, x) < 1:
        g = lam * w - y * x
    else:
        g = lam * w
    return w - lr * g
```

</details>